In [1]:
%load_ext dotenv
%dotenv

In [2]:
import pandas as pd
import os
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv, find_dotenv
import pinecone
from sentence_transformers import SentenceTransformer

/opt/anaconda3/envs/database_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
files = pd.read_csv("course_descriptions.csv", encoding = "utf-8")

In [10]:
def create_course_desciption(row):
    return f'''The Course Name is {row["course_name"]}, the slug is {row["course_slug"]}, 
    the technology is {row["course_technology"]}, and the course topic is {row["course_topic"]}.'''

In [11]:
files["course_description_new"] = files.apply(create_course_desciption, axis=1)
print(files["course_description_new"])

0      The Course Name is Introduction to Tableau, th...
1      The Course Name is The Complete Data Visualiza...
2      The Course Name is Introduction to R Programmi...
3      The Course Name is Data Preprocessing with Num...
4      The Course Name is Introduction to Data and Da...
                             ...                        
101    The Course Name is Intro to NLP for AI, the sl...
102    The Course Name is Data Analysis with ChatGPT,...
103    The Course Name is ChatGPT for Data Science, t...
104    The Course Name is Intro to LLMs, the slug is ...
105    The Course Name is Growth Analysis with SQL, P...
Name: course_description_new, Length: 106, dtype: str


In [21]:
pc = Pinecone(
    api_key=os.environ.get("PINECONE_API_KEY"),
    environment=os.environ.get("PINECONE_ENV"),
)

In [22]:
index_name = "my-index"
dimension = 384
metric = "cosine"

In [23]:
if index_name in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f"{index_name} succesfully deleted.")
else:
    print(f"{index_name} not in index list.")

my-index not in index list.


In [24]:
pc.create_index(
    name=index_name,
    dimension=dimension,
    metric=metric,
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)

{
    "name": "my-index",
    "metric": "cosine",
    "host": "my-index-v23epdq.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "2025-10",


In [25]:
index = pc.Index(index_name)

## Embedding the data

In [17]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1909.67it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
def create_embeddings(row):
    combined_text = ' '.join([str(row[field]) for field in ["course_description", "course_description_new", "course_description_short"]])
    embedding = model.encode(combined_text, show_progress_bar=False)
    return embedding

In [19]:
files["embedding"] = files.apply(create_embeddings, axis=1)

In [26]:
vectors_to_upsert = [(str(row["course_name"]), row["embedding"].tolist()) for _, row in files.iterrows()]
index.upsert(vectors=vectors_to_upsert)

print("Data upserted successfully!")

Data upserted successfully!


## Semantic Search with Pinecone and Sentence Transformers

In [28]:
query = "Clustering"
query_embedding = model.encode(query, show_progress_bar=False).tolist()

In [29]:
query_results = index.query(
    vector = [query_embedding],
    top_k = 12,
    include_values = True,
)

In [30]:
query_results

QueryResponse(matches=[{'id': 'Machine Learning in Excel',
 'score': 0.361535102,
 'values': [-0.0117317727,
            -0.0264511276,
            -0.0250257365,
            -0.0169423353,
            -0.0256783813,
            -0.0225825068,
            -0.0534479506,
            -0.0510630086,
            0.00597184617,
            0.0250511579,
            -0.0368009768,
            -0.0396631695,
            0.0695045516,
            -0.0395583585,
            -0.00341991615,
            0.0407637507,
            -0.00347200851,
            -0.00189277902,
            -0.00405908935,
            -0.0652187,
            0.0792764425,
            0.0295214951,
            -0.0502012968,
            -0.0240530781,
            0.03659961,
            0.0308817029,
            0.0652035475,
            0.00612643687,
            -0.0473339297,
            -0.0358878039,
            -0.046727363,
            0.0049373256,
            0.015325468,
            0.0524632558,
            -0

In [31]:
for match in query_results["matches"]:
    print(f"Matched item ID: {match['id']}, Score: {match['score']}")

Matched item ID: Machine Learning in Excel, Score: 0.361535102
Matched item ID: Machine Learning with K-Nearest Neighbors, Score: 0.323628932
Matched item ID: Customer Churn Analysis with SQL and Tableau, Score: 0.277688026
Matched item ID: Machine Learning in Python, Score: 0.265437126
Matched item ID: Growth Analysis with SQL, Python, and Tableau  , Score: 0.256688118
Matched item ID: Linear Algebra and Feature Selection, Score: 0.253460884
Matched item ID: Customer Engagement Analysis with SQL and Tableau, Score: 0.241708755
Matched item ID: Fashion Analytics with Tableau, Score: 0.232392326
Matched item ID: Machine Learning with Support Vector Machines, Score: 0.229316711
Matched item ID: Data Analysis with Excel Pivot Tables, Score: 0.220349312
Matched item ID: Machine Learning with Naive Bayes, Score: 0.216597557
Matched item ID: Machine Learning with Ridge and Lasso Regression, Score: 0.197357178


## Using Course Sections for Semantic Search

In [34]:
files_sections = pd.read_csv("course_section_descriptions.csv", encoding="cp1252")

In [35]:
files_sections["unique_id"] = files_sections["course_id"].astype(str) + "-" + files_sections["section_id"].astype(str)

In [36]:
files_sections["metadata"] = files_sections.apply(
    lambda row: {
        "course_name": row["course_name"],
        "section_name": row["section_name"],
        "section_description": row["section_description"]
    },
    axis=1
)

In [39]:
def create_embedding(row):
    combined_text = f''' {row["course_name"]} {row["course_technology"]} {row["course_description"]} 
    {row["section_name"]} {row["section_description"]}'''
    return model.encode(combined_text, show_progress_bar=False)

In [40]:
files_sections["embedding"] = files_sections.apply(create_embedding, axis=1)

## Upserting Course Sections into Pinecone

In [63]:
index_name2 = "bert"
dimension2 = 768
metric2 = "cosine"

In [54]:
model2 = SentenceTransformer("multi-qa-distilbert-cos-v1")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2636.24it/s, Materializing param=transformer.layer.5.sa_layer_norm.weight]   


In [64]:
if index_name2 in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name2)
    print(f"{index_name2} succesfully deleted.")
else:
    print(f"{index_name2} not in index list.")

bert succesfully deleted.


In [65]:
pc.create_index(
    name=index_name2,
    dimension=dimension2,
    metric=metric2,
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)

{
    "name": "bert",
    "metric": "cosine",
    "host": "bert-v23epdq.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 768,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "2025-10",
        

In [66]:
index = pc.Index(index_name2)

In [67]:
vectors_to_upsert = [(row["unique_id"], row["embedding"].tolist(), row["metadata"]) for index, row in files_sections.iterrows()]

In [68]:
index.upsert(vectors=vectors_to_upsert)
print("Data upserted successfully!")

PineconeApiException: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Mon, 16 Feb 2026 20:04:03 GMT', 'Content-Type': 'application/json', 'Content-Length': '102', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '1887', 'x-envoy-upstream-service-time': '57', 'x-pinecone-response-duration-ms': '1889', 'server': 'envoy'})
HTTP response body: {"code":3,"message":"Vector dimension 384 does not match the dimension of the index 768","details":[]}


In [69]:
query2 = "Clustering"
query_embedding2 = model2.encode(query2, show_progress_bar=False).tolist()

In [70]:
query_results2 = index.query(
    vector = [query_embedding2],
    top_k = 12,
    include_metadata = True,
)

In [71]:
score_threshold = 0.3

In [73]:
for match in query_results2["matches"]:
    if match["score"] >= score_threshold:
        course_details = match.get("metadata", {})
        course_name = course_details.get("course_name", "N/A")
        section_name = course_details.get("section_name", "N/A")
        section_description = course_details.get("section_description", "No description available.")

        print(f"Matched Item ID: {match['id']}, Score: {match['score']}")
        print(f"Course: {course_name} \nSection: {section_name} \nDescription: {section_description}\n")